<a href="https://colab.research.google.com/github/TheGenesisAIStory/regulatory-insight-engine/blob/main/fiorellia_lora_colab_ok.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fiorell.IA LoRA Cloud Training

This notebook runs Fiorell.IA behavior tuning in Colab without changing the shared backend runtime.

Supported paths:
- clone the repo from GitHub;
- use an existing copy in Google Drive.

Recommended runtime: Colab with GPU.

## 1. Configure The Run

Set `REPO_MODE` before running the next cells:
- `github_public`
- `github_private`
- `drive_existing`

In [ ]:
from pathlib import Path

REPO_MODE = "github_public"
REPO_URL = "https://github.com/TheGenesisAIStory/regulatory-insight-engine.git"
REPO_BRANCH = "main"
REPO_DIR = Path("/content/regulatory-insight-engine")
DRIVE_REPO_DIR = Path("/content/drive/MyDrive/regulatory-insight-engine")
DRIVE_ARTIFACTS_DIR = Path("/content/drive/MyDrive/fiorellia-runs")
CONFIG_PATH = "fiorellia/training/configs/config_lora_behavior_20260421.yaml"
ENABLE_DRIVE_EXPORT = True
print({
    "REPO_MODE": REPO_MODE,
    "REPO_URL": REPO_URL,
    "REPO_BRANCH": REPO_BRANCH,
    "REPO_DIR": str(REPO_DIR),
    "DRIVE_REPO_DIR": str(DRIVE_REPO_DIR),
    "DRIVE_ARTIFACTS_DIR": str(DRIVE_ARTIFACTS_DIR),
    "CONFIG_PATH": CONFIG_PATH,
    "ENABLE_DRIVE_EXPORT": ENABLE_DRIVE_EXPORT,
})

{'REPO_MODE': 'github_public', 'REPO_URL': 'https://github.com/TheGenesisAIStory/regulatory-insight-engine.git', 'REPO_BRANCH': 'main', 'REPO_DIR': '/content/regulatory-insight-engine', 'DRIVE_REPO_DIR': '/content/drive/MyDrive/regulatory-insight-engine', 'DRIVE_ARTIFACTS_DIR': '/content/drive/MyDrive/fiorellia-runs', 'CONFIG_PATH': 'fiorellia/training/configs/config_lora_behavior_20260421.yaml', 'ENABLE_DRIVE_EXPORT': True}


## 2. Optional: Mount Google Drive

Run this if you want to keep outputs after the Colab session ends.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 3. Prepare The Repository

For a private repo, the cell will ask for a temporary GitHub token and use it only for the clone command.

In [ ]:
import os
import shutil
import subprocess
from getpass import getpass

if REPO_MODE in {"github_public", "github_private"}:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    clone_url = REPO_URL
    if REPO_MODE == "github_private":
        token = getpass("GitHub token (temporary, input hidden): ").strip()
        if not token:
            raise ValueError("A token is required for github_private mode.")
        clone_url = REPO_URL.replace("https://", f"https://{token}@")
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, clone_url, str(REPO_DIR)], check=True)
elif REPO_MODE == "drive_existing":
    if not DRIVE_REPO_DIR.exists():
        raise FileNotFoundError(f"Drive repo not found: {DRIVE_REPO_DIR}")
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    shutil.copytree(DRIVE_REPO_DIR, REPO_DIR)
else:
    raise ValueError(f"Unsupported REPO_MODE: {REPO_MODE}")

os.chdir(REPO_DIR)
print(f"repo_ready={REPO_DIR}")

repo_ready=/content/regulatory-insight-engine


In [ ]:
%%bash
python3 /tmp/fix.py
echo "All patches applied"

176 '    allow_mps = bool(config.get("allow_mps", False))'
203 '    if has_mps and allow_mps and not use_4bit:'
210 '    elif has_mps and not allow_mps:'
211 '        print("MPS available but disabled by config allow_mps=false; keeping model on CPU.")'
264 '    allow_mps = bool(config.get("allow_mps", False))'
265 '    if not allow_mps:'
267 '        print("CPU device guard: allow_mps=false, keeping PEFT model on CPU.")'
270 '    if not allow_mps and not torch.cuda.is_available():'
294 '    if not allow_mps and not __import__("torch").cuda.is_available():'
306 '    if not allow_mps and not __import__("torch").cuda.is_available():'
311 '        print(f"Trainer device guard: allow_mps=false, Trainer device={training_args.device}.")'


## 4. Install Dependencies

Colab already provides PyTorch. This cell upgrades the Python stack needed by the Fiorell.IA training script.

In [ ]:
!python -m pip install --upgrade pip
!pip install -r fiorellia/training/requirements-lora.txt
!pip install bitsandbytes

  Using cached datasets-3.6.0-py3-none-any.whl.metadata (19 kB)
  Using cached accelerate-0.34.2-py3-none-any.whl.metadata (19 kB)
Using cached datasets-3.6.0-py3-none-any.whl (491 kB)
Using cached accelerate-0.34.2-py3-none-any.whl (324 kB)
  Attempting uninstall: datasets
    Found existing installation: datasets 4.8.4
    Uninstalling datasets-4.8.4:
      Successfully uninstalled datasets-4.8.4
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.13.0
    Uninstalling accelerate-1.13.0:
      Successfully uninstalled accelerate-1.13.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [accelerate]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
trl 1.2.0 requires accelerate>=1.4.0, but you have accelerate 0.34.2 which is incompatible.
trl 1.2.0 requires datasets>=4.7.0, but you have datasets 3.6.0 which is incompatible.


## 5. Inspect Environment, Config, And Dataset

In [ ]:
import json
from pathlib import Path

import torch
import yaml

repo_root = Path.cwd()
config_path = repo_root / CONFIG_PATH
config = yaml.safe_load(config_path.read_text(encoding="utf-8"))
dataset_path = repo_root / config["dataset_path"]
output_dir = repo_root / config["output_dir"]

print(json.dumps({
    "python": os.popen("python --version").read().strip(),
    "cuda_available": torch.cuda.is_available(),
    "cuda_device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "config_path": str(config_path),
    "dataset_exists": dataset_path.exists(),
    "dataset_path": str(dataset_path),
    "output_dir": str(output_dir),
    "base_model_name": config["base_model_name"],
    "use_4bit": config.get("use_4bit"),
}, indent=2))

with dataset_path.open("r", encoding="utf-8") as handle:
    sample = json.loads(next(line for line in handle if line.strip()))
print(json.dumps(sample, ensure_ascii=False, indent=2)[:2000])

{
  "python": "Python 3.12.13",
  "cuda_available": true,
  "cuda_device": "Tesla T4",
  "config_path": "/content/regulatory-insight-engine/fiorellia/training/configs/config_lora_behavior_20260421.yaml",
  "dataset_exists": true,
  "dataset_path": "/content/regulatory-insight-engine/fiorellia/training/supervised_v1_curated_20260421.jsonl",
  "output_dir": "/content/regulatory-insight-engine/fiorellia/training/lora/fiorellia_behavior_20260421",
  "base_model_name": "Qwen/Qwen2.5-3B-Instruct",
  "use_4bit": true
}
{
  "id": "fio-v0-006-curated",
  "category": "unsupported_abstention",
  "lang": "it",
  "messages": [
    {
      "role": "system",
      "content": "Fiorell.IA risponde solo se fonti locali recuperate supportano la risposta. Se la richiesta è troppo ampia o non coperta, si astiene senza inventare contenuti o citazioni."
    },
    {
      "role": "user",
      "content": "Puoi darmi una panoramica completa di tutto IFRS 9, inclusi hedge accounting e classificazione degli str

## 6. Optional: Quick Preflight

This reuses the existing project preflight.

In [ ]:
!python fiorellia/training/preflight_check_training.py

Fiorell.IA LoRA training preflight
python=3.12.13
executable=/usr/bin/python3
virtual_env=not active

PASS python_version
PASS import torch: 2.10.0+cu128
PASS import transformers: 4.57.6
2026-04-23 20:27:44.824773: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776976064.846331    8326 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776976064.853791    8326 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776976064.872157    8326 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776976064.872206    8326 computation_placer.cc:177] computatio

## 7. Start Training

This uses the existing Fiorell.IA training script and dated config.

In [ ]:
!pip install -q "accelerate>=1.4.0" "transformers>=4.45.0" "peft>=0.13.0" "trl>=1.2.0"

In [ ]:
!python fiorellia/training/train_lora_behavior_v1.py --config $CONFIG_PATH

2026-04-23 21:04:45.828308: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776978285.863598   18157 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776978285.874999   18157 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776978285.903307   18157 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776978285.903347   18157 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776978285.903355   18157 computation_placer.cc:177] computation placer alr

## 8. Export Adapter Artifacts

This copies the final adapter to a stable export folder and prepares a zip for download.

In [ ]:
import shutil

artifact_name = output_dir.name
artifact_root = repo_root / "artifacts"
artifact_root.mkdir(parents=True, exist_ok=True)
artifact_dir = artifact_root / artifact_name
if artifact_dir.exists():
    shutil.rmtree(artifact_dir)
shutil.copytree(output_dir, artifact_dir)
zip_base = artifact_root / artifact_name
zip_path = shutil.make_archive(str(zip_base), "zip", str(artifact_root), artifact_name)
print(f"artifact_dir={artifact_dir}")
print(f"zip_path={zip_path}")

if ENABLE_DRIVE_EXPORT:
    DRIVE_ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

    # ── Cleanup: keep only the latest zip to save Drive space ──────────────
    existing_zips = sorted(DRIVE_ARTIFACTS_DIR.glob("*.zip"))
    if existing_zips:
        print(f"Removing {len(existing_zips)} old zip(s) from Drive to free space...")
        for old_zip in existing_zips:
            old_zip.unlink()
            print(f"  deleted: {old_zip.name}")

    # ── Copy ONLY the zip (not the full folder) to Drive ───────────────────
    drive_zip = DRIVE_ARTIFACTS_DIR / f"{artifact_name}.zip"
    shutil.copy2(zip_path, drive_zip)
    print(f"drive_zip={drive_zip}")
    print(f"Drive space used (only zip, no folder copy): {drive_zip.stat().st_size / 1e6:.1f} MB")

artifact_dir=/content/regulatory-insight-engine/artifacts/fiorellia_behavior_20260421
zip_path=/content/regulatory-insight-engine/artifacts/fiorellia_behavior_20260421.zip
drive_artifact_dir=/content/drive/MyDrive/fiorellia-runs/fiorellia_behavior_20260421
drive_zip=/content/drive/MyDrive/fiorellia-runs/fiorellia_behavior_20260421.zip


## 9. Download The Zip

Use this only if you want to download directly from the Colab session.

In [ ]:
from google.colab import files

files.download(str(artifact_root / f"{artifact_name}.zip"))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 10. Next Local Step

L'addestramento e' completato. Il file zip dell'adapter e' salvato su Google Drive in:
`/MyDrive/fiorellia-runs/<nome_run>.zip`

### Step A — Scarica il zip da Drive

Se non hai usato la cella "Download The Zip", scarica il file zip direttamente da [Google Drive](https://drive.google.com/drive/folders/1-cOAupbl8sovMlXFn5Q6WLADmwwrQY2f).

> **Nota spazio Drive:** la cella "Export" ora salva **solo il zip** (non la cartella) e **cancella automaticamente i vecchi zip** prima di salvarne uno nuovo. In questo modo la cartella occupa lo spazio di un solo adapter (~100 MB).

### Step B — Posiziona l'adapter in locale

Estrai il zip e metti la cartella dell'adapter sotto:
```
fiorellia/training/lora/fiorellia_behavior_20260421
```

### Step C — Scarica i file di eval da GitHub

I file di valutazione **non vanno su Drive** — sono gia' nel repository GitHub pubblico. Clonalo in locale (se non lo hai gia') oppure scarica i file direttamente:

```bash
# Opzione 1: clona il repo completo
git clone https://github.com/TheGenesisAIStory/regulatory-insight-engine.git
cd regulatory-insight-engine

# Opzione 2: scarica solo i file eval con curl
curl -O https://raw.githubusercontent.com/TheGenesisAIStory/regulatory-insight-engine/main/fiorellia/eval/prompt_harness_with_adapter.md
curl -O https://raw.githubusercontent.com/TheGenesisAIStory/regulatory-insight-engine/main/fiorellia/eval/compare_baseline_vs_adapter_20260421.md
```

### Step D — Esegui la valutazione

Seguendo nell'ordine:
1. `fiorellia/eval/prompt_harness_with_adapter.md` — testa il modello con l'adapter attivato
2. `fiorellia/eval/compare_baseline_vs_adapter_20260421.md` — confronta le risposte baseline vs adapter